In [1]:
import sys
sys.path.append('../../Simulate/')

In [2]:
import os
import random
import numpy as np
import subprocess
import multiprocessing
import threading

from Bio import SeqIO
from tqdm import tqdm
from scipy.stats import bernoulli
from typing import Dict, Union, Tuple
from threading import Lock
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from multiprocessing import Queue 

In [3]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"
ref_fasta = working_path + 'data/ref/BSB_test.fa'
outdir = working_path + "outdir2/"
prefix = 'sim'

In [4]:
import importlib
import BSReadSim_queue
importlib.reload(BSReadSim_queue)

from BSReadSim_queue import BSReadSim
from DataProcessor_thread import DataProcessor
from LockedIterator import LockedIterator
from SetMethylation import SetMethylation
from StreamReads import StreamReads
from StreamHTSIM import StreamHTSIM
from SetExperiment import SetExperiment
from ReadProcessor import ReadProcessor

In [19]:
self = BSReadSim(ref_fasta = ref_fasta, outdir=outdir, prefix=prefix, overwrite_db=True, num_reads=10**4,
                 gzip=False, shuffle=False)
cmd_part = [self.htsim_path, self.ref_fasta] + [str(item) for key_val in self.htsim_opts.items() for item in key_val]

contig_id= 'chr10'
sim_cmd  = cmd_part + ['-c', contig_id] + ['-n', str(self.count_dict[contig_id])]

read_gen = LockedIterator(StreamHTSIM(sim_cmd=sim_cmd, pair_end=self.pair_end)) # only output 1 header for -c TODO:
var_contig, sim_data= next(read_gen)                                    # the first element of generator is variants
self.curr_contig= var_contig                                            # update the profiles
self.var_profile= self.meth_set.set_var_meth(var_contig, sim_data)      # a dict, can be empty
pos_map, meth_arr, _  = self.meth_db.load_contig(var_contig)                              # [pos_map, meth_arr, status]

self.processor  = ReadProcessor(meth_arr= meth_arr,
                                pos_map = pos_map,
                                var_profile = self.var_profile,
                                experiment  = self.experiment)

Initiating experiment...
Initiating methylation profile...

[Initiating meth_db] for chr10...
Filling with beta distribution for chr10...
Processed 187408 sites from contig chr10

[Initiating meth_db] for chr11...
Filling with beta distribution for chr11...
Processed 187844 sites from contig chr11

[Initiating meth_db] for chr12...
Filling with beta distribution for chr12...
Processed 184249 sites from contig chr12

[Initiating meth_db] for chr13...
Filling with beta distribution for chr13...
Processed 142075 sites from contig chr13

[Initiating meth_db] for chr14...
Filling with beta distribution for chr14...
Processed 154158 sites from contig chr14

[Initiating meth_db] for chr15...
Filling with beta distribution for chr15...
Processed 2252 sites from contig chr15


Simulating whole genome reads:
Reference genome file: /home/wbguo/iproject/BSReadSim/test/data/ref/BSB_test.fa
[main] Calculating the total length and effective length of the reference sequences...
[main] Contig chr10 specified, contig length: 423500, effective length: 423500
[main] No VCF input, will generate SNP randomly if mutation rate is nonzero
[htsim] seed = 1679019778
[sim_core] contig 'chr10': simulate 2158 reads...


In [20]:
self.data_processor = DataProcessor(read_gen=read_gen, n_workers=2,
                                    processor=self.processor, fastq_out=self.fastq_out)

In [21]:
# importing cProfile
import cProfile


def f():
    print("hello")

cProfile.run('self.data_processor.start_processing_mt()')

[sim_core] Generated 2158 read pairs, with 188 contain SNP, 64 contain INDEL


         54823 function calls in 14.896 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000    0.000    0.000 <frozen importlib._bootstrap>:1017(_handle_fromlist)
        5    0.000    0.000    0.000    0.000 <frozen importlib._bootstrap>:389(parent)
        1    0.000    0.000   14.896   14.896 <string>:1(<module>)
        1    0.010    0.010   14.896   14.896 DataProcessor_thread.py:18(start_processing_mt)
        2    0.000    0.000    0.000    0.000 __init__.py:36(__init__)
        2    0.000    0.000    0.001    0.001 __init__.py:43(start)
        6    0.000    0.000    0.000    0.000 _weakrefset.py:81(add)
        2    0.000    0.000    0.000    0.000 connection.py:117(__init__)
        1    0.000    0.000    0.000    0.000 connection.py:516(Pipe)
        1    0.000    0.000    0.001    0.001 context.py:110(SimpleQueue)
        3    0.000    0.000    0.000    0.000 context.py:187(get_context)
 